# Notebook 21 — Hyperparameter Tuning (Optuna) + Ensemble روی معماری v4

**پایه:** `15_Physics_Informed_IFC_MAX_Phase_Phonopy.ipynb` (v4، MAE فرکانس = 0.909 THz)

## زمینه
مسیرهای factorization (notebooks 16-18) و augmentation فیزیکی (notebooks 19-20) هر دو رد شدند.
تصمیم پروژه: بازگشت به معماری v4 بدون تغییر ساختاری، و پیشروی در فاز ۴ نقشه‌راه
(Hyperparameter Tuning + Ensemble Methods).

## هدف این Notebook
1. **بخش A — Hyperparameter Tuning با Optuna**: جست‌وجو روی نرخ یادگیری، وزن‌های Loss فیزیکی
   (`λ_symmetry`, `λ_asr`)، weight decay، اندازه‌ی batch، تعداد لایه‌های message-passing،
   و dropout — با معیار ارزیابی MAE فرکانس واقعی (محاسبه‌شده با موتور Phonopy، نه IFC MSE خام).
2. **بخش B — Ensemble**: آموزش کامل چند مدل (بهترین ترکیب‌های Optuna + چند seed متفاوت)
   و میانگین‌گیری پیش‌بینی‌ها (هم روی سطح IFC، هم روی سطح فرکانس نهایی) برای کاهش واریانس.

⚠️ **نکته‌ی مهم روش‌شناختی:** به‌خاطر یافته‌ی notebook 20 (ناسازگاری augmentation مبتنی بر
بازچینش اتم با pooling permutation-invariant)، اینجا **هیچ augmentation دیتایی** اعمال
نمی‌شود. Ensemble در این notebook فقط از طریق seed های مختلف اولیه‌سازی وزن + تنوع
هایپرپارامتر به‌دست می‌آید، نه از طریق دیتای اضافه.

## معیار مقایسه
| baseline | MAE فرکانس (THz) |
|---|---|
| v4 (notebook 15، بدون tuning) | **0.909** |
| هدف این notebook | کمتر از 0.909 |


In [1]:
!pip install -q phonopy torch_geometric optuna
print("Installed: phonopy, torch_geometric, optuna")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 787.7/787.7 kB 25.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 54.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 60.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 962.5/962.5 kB 48.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 7.4 MB/s eta 0:00:00
Installed: phonopy, torch_geometric, optuna


In [2]:
import os
import json
import yaml
import copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data, Batch
from torch_geometric.nn import Set2Set, global_mean_pool, global_max_pool
from torch_geometric.utils import softmax

from scipy.spatial.distance import cdist
from tqdm.notebook import tqdm

from phonopy import Phonopy
from phonopy.structure.atoms import PhonopyAtoms
from phonopy.file_IO import parse_FORCE_CONSTANTS

import optuna
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner

device = torch.device('cpu')
print(f"Device: {device}")
print(f"Optuna version: {optuna.__version__}")

Device: cpu
Optuna version: 4.8.0


## 1. Data paths (same as notebook 15)

In [3]:
FC_DIR       = '/kaggle/input/datasets/metisa81/pgcnndata/extracted_force_constant_All/extracted_force_constant_C'
POSCAR_DIR   = '/kaggle/input/datasets/metisa81/pgcnndata/all_poscar/all_poscar'
BANDS_DIR    = '/kaggle/input/datasets/metisa81/pgcnndata/extracted_bands_C/extracted_bands_C'
ELASTIC_FILE = '/kaggle/input/datasets/metisa81/features-dataset/mechanical_data_fixed.json'
FEATURE_CSV  = '/kaggle/input/datasets/metisa81/feature-dataset-split/5_geometry_radius_volume.csv'

for name, path in [('FC_DIR', FC_DIR), ('POSCAR_DIR', POSCAR_DIR),
                    ('BANDS_DIR', BANDS_DIR), ('ELASTIC_FILE', ELASTIC_FILE),
                    ('FEATURE_CSV', FEATURE_CSV)]:
    exists = 'OK' if os.path.exists(path) else 'MISSING'
    print(f"[{exists}]  {name}  ->  {path}")

[OK]  FC_DIR  ->  /kaggle/input/datasets/metisa81/pgcnndata/extracted_force_constant_All/extracted_force_constant_C
[OK]  POSCAR_DIR  ->  /kaggle/input/datasets/metisa81/pgcnndata/all_poscar/all_poscar
[OK]  BANDS_DIR  ->  /kaggle/input/datasets/metisa81/pgcnndata/extracted_bands_C/extracted_bands_C
[OK]  ELASTIC_FILE  ->  /kaggle/input/datasets/metisa81/features-dataset/mechanical_data_fixed.json
[OK]  FEATURE_CSV  ->  /kaggle/input/datasets/metisa81/feature-dataset-split/5_geometry_radius_volume.csv


## 2. Supercell auto-discovery + Phonopy loader (identical to notebook 15)

In [4]:
def find_correct_supercell_dim(unitcell, fc, q_test, real_freqs_test, n2_expected):
    """
    Find the correct diagonal supercell_matrix [[a,0,0],[0,b,0],[0,0,c]]
    by matching frequencies at a non-Gamma q-point.
    Returns: (a,b,c) or None if nothing matched.
    """
    n_images = n2_expected // len(unitcell.symbols)

    candidates = []
    for a in range(1, n_images + 1):
        if n_images % a != 0:
            continue
        rem = n_images // a
        for b in range(1, rem + 1):
            if rem % b != 0:
                continue
            c = rem // b
            candidates.append((a, b, c))

    best_dim, best_err = None, np.inf
    for (a, b, c) in candidates:
        try:
            phonon = Phonopy(unitcell, supercell_matrix=[[a,0,0],[0,b,0],[0,0,c]])
            if len(phonon.supercell.symbols) != n2_expected:
                continue
            phonon.force_constants = fc
            phonon.run_qpoints([q_test])
            freqs = np.sort(phonon.get_qpoints_dict()['frequencies'][0])
            if len(freqs) != len(real_freqs_test):
                continue
            err = np.max(np.abs(freqs - real_freqs_test))
            if err < best_err:
                best_err = err
                best_dim = (a, b, c)
        except Exception:
            continue

    return best_dim, best_err


def load_material_with_phonopy(yaml_path, fc_path):
    """Load one material with Phonopy: discover correct dim + build a ready Phonopy object."""
    with open(yaml_path) as f:
        data = yaml.safe_load(f)

    lattice = np.array(data['lattice'])
    symbols = [p['symbol'] for p in data['points']]
    frac_coords = np.array([p['coordinates'] for p in data['points']])
    masses = [p['mass'] for p in data['points']]
    n1 = len(symbols)

    fc = parse_FORCE_CONSTANTS(str(fc_path))
    n2 = fc.shape[1]
    if fc.shape[0] != n1:
        return None

    unitcell = PhonopyAtoms(symbols=symbols, cell=lattice, scaled_positions=frac_coords)

    segment_len = data['segment_nqpoint'][0]
    q_idx = min(segment_len // 2, len(data['phonon']) - 1)
    q_test = data['phonon'][q_idx]['q-position']
    real_freqs_test = np.sort([b['frequency'] for b in data['phonon'][q_idx]['band']])

    dim, err = find_correct_supercell_dim(unitcell, fc, q_test, real_freqs_test, n2)
    if dim is None or err > 0.01:
        return None

    phonon = Phonopy(unitcell, supercell_matrix=[[dim[0],0,0],[0,dim[1],0],[0,0,dim[2]]])
    phonon.force_constants = fc

    return {
        'phonon': phonon,
        'unitcell': unitcell,
        'symbols': symbols,
        'frac_coords': frac_coords,
        'lattice': lattice,
        'masses': masses,
        'supercell_dim': dim,
        'dim_match_error': err,
        'force_constants': fc,
        'all_q_points': [p['q-position'] for p in data['phonon']],
        'all_real_freqs': np.array([sorted([b['frequency'] for b in p['band']]) for p in data['phonon']]),
    }

print("Supercell discovery + Phonopy loader ready")

Supercell discovery + Phonopy loader ready


## 3. Build full dataset (358 materials, same filters as notebook 15)

In [5]:
POSCAR_DIR_PATH = Path(POSCAR_DIR)
FC_DIR_PATH = Path(FC_DIR)
BANDS_DIR_PATH = Path(BANDS_DIR)

fc_files = {f.stem: f for f in FC_DIR_PATH.glob('*.fc')}
band_yaml_files = {f.stem: f for f in BANDS_DIR_PATH.glob('*.yaml')}
poscar_files = {f.stem: f for f in POSCAR_DIR_PATH.glob('*.psc')}

common = sorted(set(fc_files) & set(band_yaml_files) & set(poscar_files))
print(f"Materials with .fc + .yaml + .psc: {len(common)}")

with open(ELASTIC_FILE) as f:
    elastic_json = json.load(f)
ELASTIC_KEYS = ['C11','C12','C13','C33','C44','C66',
                'B_Hill','G_Hill','E_Hill','Poisson_Hill',
                'Pugh_ratio','Debye_temperature']
elastic_data = {}
for entry in elastic_json:
    mat = entry.get('material', '')
    vals = [float(entry.get(k, 0) or 0) for k in ELASTIC_KEYS]
    elastic_data[mat] = np.array(vals, dtype=np.float32)

Materials with .fc + .yaml + .psc: 358


In [6]:
MAX_ATOMS_FOR_IFC = 12

raw_samples = []
failed_phonopy = []
failed_other = []

for formula in tqdm(common, desc="Loading materials with Phonopy"):
    try:
        with open(band_yaml_files[formula]) as f:
            quick_check = yaml.safe_load(f)
        if quick_check['natom'] > MAX_ATOMS_FOR_IFC:
            continue

        result = load_material_with_phonopy(band_yaml_files[formula], fc_files[formula])
        if result is None:
            failed_phonopy.append(formula)
            continue

        positions_cart = result['frac_coords'] @ result['lattice']

        raw_samples.append({
            'formula':           formula,
            'n_atoms':           len(result['symbols']),
            'lattice':           result['lattice'].astype(np.float32),
            'positions':         positions_cart.astype(np.float32),
            'atom_elements':     result['symbols'],
            'masses':            np.array(result['masses'], dtype=np.float32),
            'force_constants':   result['force_constants'].astype(np.float32),
            'supercell_dim':     result['supercell_dim'],
            'elastic_constants': elastic_data.get(formula, np.zeros(len(ELASTIC_KEYS), np.float32)),
            'y_phonon':          result['all_real_freqs'].astype(np.float32),
            'phonon_obj':        result['phonon'],
        })
    except Exception as e:
        failed_other.append((formula, str(e)))

print(f"\nOK: {len(raw_samples)} | failed_phonopy: {len(failed_phonopy)} | failed_other: {len(failed_other)}")

Loading materials with Phonopy:   0%|          | 0/358 [00:00<?, ?it/s]

/tmp/ipykernel_58/1747253744.py:28: DeprecationWarning: get_qpoints_dict() is deprecated. Use the qpoints property to access the Qpoints result object; its frequencies, eigenvectors, group_velocities, and dynamical_matrices attributes replace the dict keys.
  freqs = np.sort(phonon.get_qpoints_dict()['frequencies'][0])
/usr/local/lib/python3.12/dist-packages/phonopy/api_phonopy.py:278: UserWarning: Warning: Point group symmetries of supercell and primitivecell are different.
  self._search_primitive_symmetry()



OK: 358 | failed_phonopy: 0 | failed_other: 0


## 4. Train / Val / Test split

⚠️ **حیاتی**: از همان `RANDOM_SEED=42` نسخه v4 استفاده می‌شود تا split دقیقاً یکسان باشد
و مقایسه‌ی MAE با baseline (0.909) معتبر بماند.

In [7]:
RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)
idx = rng.permutation(len(raw_samples))

n_total = len(raw_samples)
n_tr = int(0.8 * n_total)
n_va = int(0.1 * n_total)

train_idx = idx[:n_tr]
val_idx   = idx[n_tr:n_tr+n_va]
test_idx  = idx[n_tr+n_va:]

print(f"Train: {len(train_idx)} | Val: {len(val_idx)} | Test: {len(test_idx)}")

Train: 286 | Val: 35 | Test: 37


## 5. Graph construction (Bond + Atom graphs) — identical to notebook 15

In [8]:
CUTOFF = 8.0

df_raw = pd.read_csv(FEATURE_CSV)
symbol_col = next((c for c in df_raw.columns if c.lower() in ['symbol', 'element', 'atom']), df_raw.columns[0])
feature_cols = [c for c in df_raw.columns if c not in [symbol_col, 'atomic_number']]
feature_cols = [c for c in feature_cols if pd.api.types.is_numeric_dtype(df_raw[c])]
df_features = df_raw[feature_cols].copy()
df_features.index = df_raw[symbol_col]
N_ATOM_FEATURES = len(feature_cols)
print(f"Atomic features: {N_ATOM_FEATURES} -> {feature_cols}")


def structure_to_bond_graph(positions, n_atoms_sample):
    n = len(positions)
    distances = cdist(positions, positions)
    bonds = [(i, j) for i in range(n) for j in range(i+1, n) if distances[i, j] <= CUTOFF]

    if len(bonds) == 0:
        x = torch.zeros((1, 6), dtype=torch.float)
        edge_index = torch.zeros((2, 0), dtype=torch.long)
        edge_attr = torch.zeros((0, 1), dtype=torch.float)
        u = torch.tensor([[1.0, 1.0, 1.0]], dtype=torch.float)
        return Data(x=x, edge_index=edge_index, edge_attr=edge_attr, u=u, n_atoms=torch.tensor([n_atoms_sample]))

    node_features = []
    for i, j in bonds:
        d = distances[i, j]
        coord_i = np.sum(distances[i] <= CUTOFF) - 1
        coord_j = np.sum(distances[j] <= CUTOFF) - 1
        node_features.append([d, coord_i, coord_j, (coord_i + coord_j) / 2, 0.0, 0.0])
    x = torch.tensor(node_features, dtype=torch.float)

    edge_index, edge_attr = [], []
    for idx1, (i1, j1) in enumerate(bonds):
        for idx2, (i2, j2) in enumerate(bonds[idx1+1:]):
            idx2 += idx1 + 1
            shared = set([i1, j1]) & set([i2, j2])
            if len(shared) == 1:
                s = shared.pop()
                a = positions[i1 if i1 != s else j1] - positions[s]
                b = positions[i2 if i2 != s else j2] - positions[s]
                na, nb = np.linalg.norm(a), np.linalg.norm(b)
                if na > 0 and nb > 0:
                    cos_angle = np.clip(np.dot(a, b) / (na * nb), -1.0, 1.0)
                    angle = np.arccos(cos_angle)
                    edge_index.extend([[idx1, idx2], [idx2, idx1]])
                    edge_attr.extend([[angle], [angle]])

    edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous() if edge_index else torch.zeros((2, 0), dtype=torch.long)
    edge_attr = torch.tensor(edge_attr, dtype=torch.float) if edge_attr else torch.zeros((0, 1), dtype=torch.float)
    u = torch.tensor([[1.0, 1.0, 1.0]], dtype=torch.float)
    return Data(x=x, edge_index=edge_index, edge_attr=edge_attr, u=u, n_atoms=torch.tensor([n_atoms_sample]))


def structure_to_atom_graph(atom_elements, positions, n_atoms_sample):
    node_features = []
    for elem in atom_elements:
        if elem in df_features.index:
            feats = df_features.loc[elem].values.astype(np.float32)
        else:
            feats = np.zeros(N_ATOM_FEATURES, dtype=np.float32)
        node_features.append(feats)
    x = torch.tensor(np.array(node_features), dtype=torch.float)

    distances = cdist(positions, positions)
    edge_index, edge_attr = [], []
    n = len(positions)
    for i in range(n):
        for j in range(n):
            if i != j and distances[i, j] <= CUTOFF:
                edge_index.append([i, j])
                edge_attr.append([distances[i, j]])

    edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()
    edge_attr = torch.tensor(edge_attr, dtype=torch.float)
    u = torch.tensor([[1.0]], dtype=torch.float)
    return Data(x=x, edge_index=edge_index, edge_attr=edge_attr, u=u, n_atoms=torch.tensor([n_atoms_sample]))

print("Graph construction functions ready")

Atomic features: 7 -> ['atomic_weight', 'atomic_radius', 'atomic_radius_rahm', 'covalent_radius_cordero', 'vdw_radius', 'atomic_volume', 'lattice_constant']
Graph construction functions ready


In [9]:
bond_graphs, atom_graphs, ifc_targets = [], [], []

for s in tqdm(raw_samples, desc="Building graphs"):
    bond_graphs.append(structure_to_bond_graph(s['positions'], s['n_atoms']))
    atom_graphs.append(structure_to_atom_graph(s['atom_elements'], s['positions'], s['n_atoms']))
    ifc_targets.append(s['force_constants'])

MAX_ATOMS = max(s['n_atoms'] for s in raw_samples)
print(f"Graphs built: {len(bond_graphs)} | max_atoms = {MAX_ATOMS}")

Building graphs:   0%|          | 0/358 [00:00<?, ?it/s]

Graphs built: 358 | max_atoms = 8


## 6. Parameterized model architecture

معماری پایه دقیقاً همان `DualGraphIFCNet` نسخه v4 است، فقط این هایپرپارامترها اکنون
قابل‌تنظیم‌اند (به‌جای hard-code):
- `hidden_dim`
- `n_bond_layers`, `n_atom_layers` (تعداد لایه‌های message passing)
- `dropout`
- `set2set_steps`

In [10]:
class CustomMessagePassing(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.attention_mlp = nn.Sequential(
            nn.Linear(3 * hidden_dim, hidden_dim // 2),
            nn.LayerNorm(hidden_dim // 2),
            nn.SiLU(),
            nn.Linear(hidden_dim // 2, 1),
            nn.LeakyReLU(0.2)
        )
        self.message_mlp = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.SiLU(),
        )

    def forward(self, x, edge_index, edge_attr):
        source_nodes = edge_index[0]
        target_nodes = edge_index[1]
        neighbor_features = x[source_nodes]
        target_features = x[target_nodes]

        attention_input = torch.cat([target_features, neighbor_features, edge_attr], dim=1)
        attention_scores = self.attention_mlp(attention_input)
        attention_weights = softmax(attention_scores, target_nodes, num_nodes=x.size(0))

        messages = neighbor_features * edge_attr
        weighted_messages = messages * attention_weights

        num_nodes = x.size(0)
        aggregated = torch.zeros(num_nodes, self.hidden_dim, device=x.device)
        aggregated.index_add_(0, target_nodes, weighted_messages)

        return self.message_mlp(aggregated)


class DualGraphIFCNet(nn.Module):
    def __init__(self, n_bond_features=6, n_atom_features=7, edge_dim=1, max_atoms=12,
                 hidden_dim=128, n_bond_layers=5, n_atom_layers=2, dropout=0.1,
                 set2set_steps=1):
        super().__init__()
        self.max_atoms = max_atoms
        hidden = hidden_dim

        self.bond_embedding = nn.Sequential(
            nn.Linear(n_bond_features, hidden), nn.BatchNorm1d(hidden), nn.SiLU(), nn.Dropout(dropout))
        self.bond_edge_embedding = nn.Sequential(nn.Linear(edge_dim, hidden), nn.SiLU())
        self.bond_message_layers = nn.ModuleList([CustomMessagePassing(hidden) for _ in range(n_bond_layers)])
        self.bond_layer_norms = nn.ModuleList([nn.LayerNorm(hidden) for _ in range(n_bond_layers)])
        self.bond_attention = nn.Sequential(
            nn.Linear(hidden, hidden // 4), nn.SiLU(), nn.Linear(hidden // 4, 1), nn.Sigmoid())

        self.atom_embedding = nn.Sequential(
            nn.Linear(n_atom_features, hidden), nn.BatchNorm1d(hidden), nn.SiLU(), nn.Dropout(dropout))
        self.atom_edge_embedding = nn.Sequential(nn.Linear(edge_dim, hidden), nn.SiLU())
        self.atom_message_layers = nn.ModuleList([CustomMessagePassing(hidden) for _ in range(n_atom_layers)])
        self.atom_layer_norms = nn.ModuleList([nn.LayerNorm(hidden) for _ in range(n_atom_layers)])
        self.atom_attention = nn.Sequential(
            nn.Linear(hidden, hidden // 4), nn.SiLU(), nn.Linear(hidden // 4, 1), nn.Sigmoid())

        self.bond_residual_weight = nn.Parameter(torch.tensor(0.3))
        self.atom_residual_weight = nn.Parameter(torch.tensor(0.3))

        self.set2set_pool = Set2Set(hidden, processing_steps=set2set_steps)
        self.mean_pool = global_mean_pool
        self.max_pool = global_max_pool
        self.global_mlp = nn.Sequential(nn.Linear(3, hidden // 4), nn.SiLU())

        combined_dim = 8 * hidden + hidden // 4
        ifc_out_dim = max_atoms * max_atoms * 3 * 3

        self.ifc_head = nn.Sequential(
            nn.Linear(combined_dim, 512), nn.LayerNorm(512), nn.SiLU(), nn.Dropout(dropout * 2),
            nn.Linear(512, 256), nn.LayerNorm(256), nn.SiLU(), nn.Dropout(dropout * 2),
            nn.Linear(256, ifc_out_dim)
        )

    def _encode_graph(self, x, edge_index, edge_attr, embedding, edge_embedding,
                       message_layers, layer_norms, attention, residual_weight):
        x = embedding(x)
        edge_feat = edge_embedding(edge_attr)
        for i, (msg, ln) in enumerate(zip(message_layers, layer_norms)):
            x_res = x
            x = msg(x, edge_index, edge_feat)
            x = ln(x)
            if i > 0:
                x = x + residual_weight * x_res
            x = F.silu(x)
        attn = attention(x)
        return x * attn, x

    def forward(self, bond_data, atom_data):
        bond_w, x_bond = self._encode_graph(
            bond_data.x, bond_data.edge_index, bond_data.edge_attr,
            self.bond_embedding, self.bond_edge_embedding,
            self.bond_message_layers, self.bond_layer_norms,
            self.bond_attention, self.bond_residual_weight)

        bond_set2set = self.set2set_pool(bond_w, bond_data.batch)
        bond_mean = self.mean_pool(x_bond, bond_data.batch)
        bond_max = self.max_pool(x_bond, bond_data.batch)

        atom_w, x_atom = self._encode_graph(
            atom_data.x, atom_data.edge_index, atom_data.edge_attr,
            self.atom_embedding, self.atom_edge_embedding,
            self.atom_message_layers, self.atom_layer_norms,
            self.atom_attention, self.atom_residual_weight)

        atom_set2set = self.set2set_pool(atom_w, atom_data.batch)
        atom_mean = self.mean_pool(x_atom, atom_data.batch)
        atom_max = self.max_pool(x_atom, atom_data.batch)

        global_features = self.global_mlp(bond_data.u)

        combined = torch.cat([
            bond_set2set, bond_mean, bond_max,
            atom_set2set, atom_mean, atom_max,
            global_features
        ], dim=1)

        ifc_flat = self.ifc_head(combined)
        batch_size = ifc_flat.shape[0]
        ifc_pred = ifc_flat.view(batch_size, self.max_atoms, self.max_atoms, 3, 3)
        return ifc_pred

print("Parameterized DualGraphIFCNet ready")

Parameterized DualGraphIFCNet ready


## 7. Physics-informed losses (weights now tunable) + training loop

In [11]:
def symmetry_loss(ifc_pred, n_atoms_real):
    total = 0.0
    batch_size = ifc_pred.shape[0]
    for b in range(batch_size):
        n = n_atoms_real[b]
        ifc = ifc_pred[b, :n, :n, :, :]
        ifc_T = ifc.permute(1, 0, 3, 2)
        total = total + torch.mean((ifc - ifc_T) ** 2)
    return total / batch_size


def acoustic_sum_rule_loss(ifc_pred, n_atoms_real):
    total = 0.0
    batch_size = ifc_pred.shape[0]
    for b in range(batch_size):
        n = n_atoms_real[b]
        ifc = ifc_pred[b, :n, :n, :, :]
        asr = torch.sum(ifc, dim=1)
        total = total + torch.mean(asr ** 2)
    return total / batch_size


def compute_batch_loss(model, bond_data, atom_data, ifc_target_list, elements_list, device,
                        lambda_symmetry, lambda_asr):
    ifc_pred = model(bond_data, atom_data)
    batch_size = ifc_pred.shape[0]
    n_atoms_real = torch.tensor([len(e) for e in elements_list], dtype=torch.long)

    mse_total = 0.0
    for b in range(batch_size):
        n = n_atoms_real[b]
        pred_ifc_b = ifc_pred[b, :n, :n, :, :]
        true_ifc_full = torch.tensor(ifc_target_list[b], dtype=torch.float32, device=device)
        true_ifc_b = true_ifc_full[:, :n, :, :]
        mse_total = mse_total + F.mse_loss(pred_ifc_b, true_ifc_b)

    mse_total = mse_total / batch_size

    sym_loss = symmetry_loss(ifc_pred, n_atoms_real)
    asr_loss = acoustic_sum_rule_loss(ifc_pred, n_atoms_real)

    total_loss = mse_total + lambda_symmetry * sym_loss + lambda_asr * asr_loss

    return total_loss, mse_total.item(), sym_loss.item(), asr_loss.item(), ifc_pred


def make_batches(indices, batch_size, shuffle=False):
    idx_arr = np.array(indices)
    if shuffle:
        idx_arr = np.random.permutation(idx_arr)
    for i in range(0, len(idx_arr), batch_size):
        yield idx_arr[i:i+batch_size]


def run_epoch(model, indices, optimizer, batch_size, shuffle, lambda_symmetry, lambda_asr):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss, total_mse = 0., 0.
    n_batches = 0

    for batch_idx in make_batches(indices, batch_size, shuffle):
        bond_list = [bond_graphs[i] for i in batch_idx]
        atom_list = [atom_graphs[i] for i in batch_idx]
        ifc_list = [ifc_targets[i] for i in batch_idx]
        elem_list = [raw_samples[i]['atom_elements'] for i in batch_idx]

        bond_batch = Batch.from_data_list(bond_list).to(device)
        atom_batch = Batch.from_data_list(atom_list).to(device)

        with torch.set_grad_enabled(is_train):
            if is_train:
                optimizer.zero_grad()
            loss, mse_v, sym_v, asr_v, _ = compute_batch_loss(
                model, bond_batch, atom_batch, ifc_list, elem_list, device,
                lambda_symmetry, lambda_asr)
            if is_train:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

        total_loss += loss.item()
        total_mse += mse_v
        n_batches += 1

    n_batches = max(n_batches, 1)
    return total_loss/n_batches, total_mse/n_batches

print("Loss functions + run_epoch ready")

Loss functions + run_epoch ready


## 8. Real frequency MAE evaluation (via Phonopy engine)

همان تابع ارزیابی نسخه v4 (نه فرمول دستی) — این معیار نهایی است که هم برای انتخاب
هایپرپارامتر در Optuna و هم برای ارزیابی نهایی ensemble استفاده می‌شود.

In [12]:
def predict_frequencies_with_phonopy(ifc_pred_np, sample):
    phonon = sample['phonon_obj']
    n = sample['n_atoms']
    ifc_for_phonopy = ifc_pred_np[:n, :n, :, :]
    fc_copy = sample['force_constants'].copy()
    fc_copy[:n, :n, :, :] = ifc_for_phonopy
    phonon.force_constants = fc_copy
    q_points = sample.get('all_q_points_subset', [[0, 0, 0]])
    phonon.run_qpoints(q_points)
    freqs = phonon.get_qpoints_dict()['frequencies']
    return np.array(freqs)


def evaluate_freq_mae(model, indices, return_ifc_preds=False):
    """Real frequency MAE (Gamma point, |max frequency| comparison, same protocol as v4)."""
    model.eval()
    all_errors = []
    ifc_preds_by_idx = {}

    with torch.no_grad():
        for i in indices:
            s = raw_samples[i]
            bond_batch = Batch.from_data_list([bond_graphs[i]]).to(device)
            atom_batch = Batch.from_data_list([atom_graphs[i]]).to(device)
            ifc_pred = model(bond_batch, atom_batch)[0].cpu().numpy()

            if return_ifc_preds:
                ifc_preds_by_idx[i] = ifc_pred

            try:
                s['all_q_points_subset'] = [[0, 0, 0]]
                pred_freqs = predict_frequencies_with_phonopy(ifc_pred, s)
                true_peak = float(np.max(s['y_phonon']))
                pred_peak = float(np.max(np.abs(pred_freqs)))
                all_errors.append(abs(true_peak - pred_peak))
            except Exception:
                continue

    mae = float(np.mean(all_errors)) if all_errors else float('nan')
    if return_ifc_preds:
        return mae, ifc_preds_by_idx
    return mae

print("Frequency evaluation (Phonopy engine) ready")

Frequency evaluation (Phonopy engine) ready


## بخش A — Hyperparameter Tuning با Optuna

### فضای جست‌وجو
| هایپرپارامتر | بازه | نوع |
|---|---|---|
| `lr` | 1e-4 تا 3e-3 (log) | float |
| `weight_decay` | 1e-6 تا 1e-3 (log) | float |
| `lambda_symmetry` | 0.0 تا 0.3 | float |
| `lambda_asr` | 0.0 تا 0.3 | float |
| `batch_size` | {4, 8, 16} | categorical |
| `hidden_dim` | {64, 96, 128} | categorical |
| `n_bond_layers` | {3, 4, 5, 6} | int |
| `n_atom_layers` | {1, 2, 3} | int |
| `dropout` | 0.0 تا 0.3 | float |
| `set2set_steps` | {1, 2, 3} | int |

### معیار objective
برخلاف tuning معمول (که معمولاً روی val loss کار می‌کند)، اینجا مستقیماً از **MAE فرکانس
واقعی** (محاسبه‌شده با Phonopy روی مجموعه‌ی validation) به‌عنوان معیار نهایی هر trial
استفاده می‌شود، چون این دقیقاً همان معیاری است که در مقاله گزارش خواهد شد. برای سرعت،
epoch های آموزش هر trial کمتر از آموزش کامل v4 است (early-stopping با patience کوچک‌تر)
و از `MedianPruner` برای قطع زودهنگام trial های ضعیف استفاده می‌شود.

⚠️ به‌خاطر هزینه‌ی بالای هر trial (آموزش + ارزیابی Phonopy)، تعداد trial ها محدود
(پیش‌فرض ۳۰) نگه داشته شده؛ در صورت وجود زمان بیشتر روی Kaggle، `N_TRIALS` را افزایش دهید.

In [13]:
N_TRIALS = 30
TUNING_EPOCHS = 150
TUNING_PATIENCE = 30
TUNING_BATCH_EVAL_EVERY = 10  # report intermediate value to Optuna every N epochs


def objective(trial):
    lr = trial.suggest_float('lr', 1e-4, 3e-3, log=True)
    weight_decay = trial.suggest_float('weight_decay', 1e-6, 1e-3, log=True)
    lambda_symmetry = trial.suggest_float('lambda_symmetry', 0.0, 0.3)
    lambda_asr = trial.suggest_float('lambda_asr', 0.0, 0.3)
    batch_size = trial.suggest_categorical('batch_size', [4, 8, 16])
    hidden_dim = trial.suggest_categorical('hidden_dim', [64, 96, 128])
    n_bond_layers = trial.suggest_int('n_bond_layers', 3, 6)
    n_atom_layers = trial.suggest_int('n_atom_layers', 1, 3)
    dropout = trial.suggest_float('dropout', 0.0, 0.3)
    set2set_steps = trial.suggest_categorical('set2set_steps', [1, 2, 3])

    torch.manual_seed(trial.number)
    np.random.seed(trial.number)

    model = DualGraphIFCNet(
        n_bond_features=6, n_atom_features=N_ATOM_FEATURES, edge_dim=1, max_atoms=MAX_ATOMS,
        hidden_dim=hidden_dim, n_bond_layers=n_bond_layers, n_atom_layers=n_atom_layers,
        dropout=dropout, set2set_steps=set2set_steps
    ).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=15)

    best_val_loss = float('inf')
    best_state = None
    patience_counter = 0

    for epoch in range(TUNING_EPOCHS):
        train_loss, _ = run_epoch(model, train_idx, optimizer, batch_size, True,
                                   lambda_symmetry, lambda_asr)
        val_loss, _ = run_epoch(model, val_idx, None, batch_size, False,
                                 lambda_symmetry, lambda_asr)
        scheduler.step(val_loss)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1

        if epoch % TUNING_BATCH_EVAL_EVERY == 0:
            trial.report(val_loss, epoch)
            if trial.should_prune():
                raise optuna.TrialPruned()

        if patience_counter >= TUNING_PATIENCE:
            break

    model.load_state_dict(best_state)
    freq_mae = evaluate_freq_mae(model, val_idx)

    trial.set_user_attr('best_val_ifc_loss', best_val_loss)
    trial.set_user_attr('n_epochs_trained', epoch + 1)

    return freq_mae


print(f"Objective function ready. N_TRIALS={N_TRIALS}, TUNING_EPOCHS={TUNING_EPOCHS}")

Objective function ready. N_TRIALS=30, TUNING_EPOCHS=150


In [14]:
study = optuna.create_study(
    direction='minimize',
    sampler=TPESampler(seed=42),
    pruner=MedianPruner(n_warmup_steps=5)
)

study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

print(f"\nBest trial: #{study.best_trial.number}")
print(f"Best val frequency MAE: {study.best_value:.4f} THz")
print(f"Best params: {study.best_params}")
print(f"\nBaseline v4 (notebook 15, no tuning): 0.909 THz")

[I 2026-07-07 00:49:11,639] A new study created in memory with name: no-name-938f32b2-9ac2-4c17-9ac1-0fc581b67d76


  0%|          | 0/30 [00:00<?, ?it/s]

/tmp/ipykernel_58/3431193008.py:10: DeprecationWarning: get_qpoints_dict() is deprecated. Use the qpoints property to access the Qpoints result object; its frequencies, eigenvectors, group_velocities, and dynamical_matrices attributes replace the dict keys.
  freqs = phonon.get_qpoints_dict()['frequencies']


[I 2026-07-07 00:55:28,903] Trial 0 finished with value: 0.906273114796232 and parameters: {'lr': 0.0003574712922600243, 'weight_decay': 0.0007114476009343421, 'lambda_symmetry': 0.21959818254342153, 'lambda_asr': 0.17959754525911098, 'batch_size': 4, 'hidden_dim': 64, 'n_bond_layers': 3, 'n_atom_layers': 3, 'dropout': 0.2497327922401265, 'set2set_steps': 1}. Best is trial 0 with value: 0.906273114796232.
[I 2026-07-07 01:03:41,236] Trial 1 finished with value: 0.9055566253279482 and parameters: {'lr': 0.0002814509271606064, 'weight_decay': 3.752055855124284e-05, 'lambda_symmetry': 0.12958350559263473, 'lambda_asr': 0.08736874205941257, 'batch_size': 4, 'hidden_dim': 128, 'n_bond_layers': 3, 'n_atom_layers': 2, 'dropout': 0.17772437065861274, 'set2set_steps': 2}. Best is trial 1 with value: 0.9055566253279482.
[I 2026-07-07 01:07:56,504] Trial 2 finished with value: 0.9066428001507838 and parameters: {'lr': 0.00012476394272569445, 'weight_decay': 0.0007025166339242157, 'lambda_symmetry

In [15]:
trials_df = study.trials_dataframe().sort_values('value')
trials_df.to_csv('/kaggle/working/optuna_trials.csv', index=False)
print(trials_df[['number', 'value', 'state'] + [c for c in trials_df.columns if c.startswith('params_')]].head(10).to_string())

    number     value   state  params_batch_size  params_dropout  params_hidden_dim  params_lambda_asr  params_lambda_symmetry  params_lr  params_n_atom_layers  params_n_bond_layers  params_set2set_steps  params_weight_decay
23      23  0.115167  PRUNED                 16        0.125128                 96           0.037783                0.004211   0.002162                     2                     5                     3             0.000084
28      28  0.142477  PRUNED                  8        0.127407                 96           0.027095                0.103444   0.002122                     1                     6                     3             0.000811
25      25  0.142509  PRUNED                 16        0.201293                 64           0.032106                0.040246   0.001524                     2                     5                     1             0.000020
12      12  0.157358  PRUNED                 16        0.137788                 96           0.093369   

## بخش B — Ensemble

### استراتژی
به‌جای اتکا به یک ترکیب هایپرپارامتر منفرد، **top-K** ترکیب برتر (پیش‌فرض K=5، از trial های
موفق Optuna) با seed های آموزشی متفاوت به‌طور کامل (با epoch و patience کامل، مثل v4)
آموزش داده می‌شوند. سپس دو استراتژی میانگین‌گیری مقایسه می‌شوند:

1. **Frequency-level averaging**: هر مدل جداگانه فرکانس نهایی را پیش‌بینی می‌کند، سپس
   میانگین فرکانس‌های پیش‌بینی‌شده گرفته می‌شود.
2. **IFC-level averaging**: ماتریس IFC پیش‌بینی‌شده‌ی همه‌ی مدل‌ها میانگین گرفته می‌شود
   و سپس **یک‌بار** از طریق Phonopy فرکانس محاسبه می‌شود.

⚠️ توجه: چون معماری permutation-invariant است (نتیجه‌ی notebook 20)، میانگین‌گیری IFC
بین مدل‌های **متفاوت** (نه بازچینش‌های یک ماده) مشکلی از جنس ناسازگاری notebook 20 ندارد —
اینجا هر مدل روی همان دیتای اصلی (بدون augmentation) آموزش دیده و فقط seed/معماری‌اش
فرق دارد.

In [16]:
TOP_K_ENSEMBLE = 5
FULL_EPOCHS = 500
FULL_PATIENCE = 100

successful_trials = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]
successful_trials.sort(key=lambda t: t.value)
top_trials = successful_trials[:TOP_K_ENSEMBLE]

print(f"Selected top {len(top_trials)} trials for full ensemble training:")
for t in top_trials:
    print(f"  Trial #{t.number}: val_freq_mae={t.value:.4f} | params={t.params}")

Selected top 5 trials for full ensemble training:
  Trial #22: val_freq_mae=0.8840 | params={'lr': 0.0022115587868729806, 'weight_decay': 9.970457472238286e-05, 'lambda_symmetry': 0.024883394129635337, 'lambda_asr': 0.0012707196720200351, 'batch_size': 16, 'hidden_dim': 96, 'n_bond_layers': 6, 'n_atom_layers': 2, 'dropout': 0.1235817918310527, 'set2set_steps': 3}
  Trial #16: val_freq_mae=0.8858 | params={'lr': 0.0016597065269683313, 'weight_decay': 0.00010706317716917264, 'lambda_symmetry': 0.10751708543716251, 'lambda_asr': 0.0015045128496013567, 'batch_size': 16, 'hidden_dim': 96, 'n_bond_layers': 6, 'n_atom_layers': 2, 'dropout': 0.1132032454041606, 'set2set_steps': 3}
  Trial #26: val_freq_mae=0.8870 | params={'lr': 0.002113807251214218, 'weight_decay': 0.0004216910458833378, 'lambda_symmetry': 0.10794329013034203, 'lambda_asr': 0.0006463647625395721, 'batch_size': 16, 'hidden_dim': 96, 'n_bond_layers': 6, 'n_atom_layers': 1, 'dropout': 0.13293087684455998, 'set2set_steps': 3}
  T

In [17]:
def train_full_model(params, seed):
    torch.manual_seed(seed)
    np.random.seed(seed)

    model = DualGraphIFCNet(
        n_bond_features=6, n_atom_features=N_ATOM_FEATURES, edge_dim=1, max_atoms=MAX_ATOMS,
        hidden_dim=params['hidden_dim'], n_bond_layers=params['n_bond_layers'],
        n_atom_layers=params['n_atom_layers'], dropout=params['dropout'],
        set2set_steps=params['set2set_steps']
    ).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=params['lr'],
                                   weight_decay=params['weight_decay'])
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=20)

    best_val_loss = float('inf')
    best_state = None
    patience_counter = 0

    for epoch in range(FULL_EPOCHS):
        train_loss, _ = run_epoch(model, train_idx, optimizer, params['batch_size'], True,
                                   params['lambda_symmetry'], params['lambda_asr'])
        val_loss, _ = run_epoch(model, val_idx, None, params['batch_size'], False,
                                 params['lambda_symmetry'], params['lambda_asr'])
        scheduler.step(val_loss)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= FULL_PATIENCE:
            break

    model.load_state_dict(best_state)
    return model, best_val_loss


ensemble_models = []
ensemble_info = []

for rank, trial in enumerate(top_trials):
    print(f"\n=== Training ensemble member {rank+1}/{len(top_trials)} (from trial #{trial.number}) ===")
    seed = 1000 + rank
    model, best_val_loss = train_full_model(trial.params, seed)
    freq_mae_solo = evaluate_freq_mae(model, test_idx)
    print(f"  Solo test frequency MAE: {freq_mae_solo:.4f} THz (best_val_ifc_loss={best_val_loss:.4f})")

    torch.save(model.state_dict(), f'/kaggle/working/ensemble_model_{rank}.pt')
    ensemble_models.append(model)
    ensemble_info.append({
        'rank': rank, 'trial_number': trial.number, 'seed': seed,
        'params': trial.params, 'solo_test_freq_mae': freq_mae_solo,
        'best_val_ifc_loss': best_val_loss
    })

print("\n=== Solo model summary ===")
for info in ensemble_info:
    print(f"  Model {info['rank']} (trial #{info['trial_number']}): test freq MAE = {info['solo_test_freq_mae']:.4f} THz")
print(f"\nBaseline v4 (single model, no tuning): 0.909 THz")


=== Training ensemble member 1/5 (from trial #22) ===


/tmp/ipykernel_58/3431193008.py:10: DeprecationWarning: get_qpoints_dict() is deprecated. Use the qpoints property to access the Qpoints result object; its frequencies, eigenvectors, group_velocities, and dynamical_matrices attributes replace the dict keys.
  freqs = phonon.get_qpoints_dict()['frequencies']


  Solo test frequency MAE: 0.8903 THz (best_val_ifc_loss=0.0254)

=== Training ensemble member 2/5 (from trial #16) ===
  Solo test frequency MAE: 0.8931 THz (best_val_ifc_loss=0.0435)

=== Training ensemble member 3/5 (from trial #26) ===
  Solo test frequency MAE: 0.8908 THz (best_val_ifc_loss=0.0398)

=== Training ensemble member 4/5 (from trial #10) ===
  Solo test frequency MAE: 0.8971 THz (best_val_ifc_loss=0.0588)

=== Training ensemble member 5/5 (from trial #21) ===
  Solo test frequency MAE: 0.8971 THz (best_val_ifc_loss=0.0506)

=== Solo model summary ===
  Model 0 (trial #22): test freq MAE = 0.8903 THz
  Model 1 (trial #16): test freq MAE = 0.8931 THz
  Model 2 (trial #26): test freq MAE = 0.8908 THz
  Model 3 (trial #10): test freq MAE = 0.8971 THz
  Model 4 (trial #21): test freq MAE = 0.8971 THz

Baseline v4 (single model, no tuning): 0.909 THz


### ارزیابی نهایی Ensemble روی مجموعه‌ی Test

In [18]:
def evaluate_ensemble_frequency_level(models, indices):
    """Average the final predicted frequency across models (per-material)."""
    all_errors = []
    for i in indices:
        s = raw_samples[i]
        pred_peaks = []
        for model in models:
            model.eval()
            with torch.no_grad():
                bond_batch = Batch.from_data_list([bond_graphs[i]]).to(device)
                atom_batch = Batch.from_data_list([atom_graphs[i]]).to(device)
                ifc_pred = model(bond_batch, atom_batch)[0].cpu().numpy()
            try:
                s['all_q_points_subset'] = [[0, 0, 0]]
                pred_freqs = predict_frequencies_with_phonopy(ifc_pred, s)
                pred_peaks.append(float(np.max(np.abs(pred_freqs))))
            except Exception:
                continue
        if not pred_peaks:
            continue
        avg_pred_peak = np.mean(pred_peaks)
        true_peak = float(np.max(s['y_phonon']))
        all_errors.append(abs(true_peak - avg_pred_peak))
    return float(np.mean(all_errors)) if all_errors else float('nan')


def evaluate_ensemble_ifc_level(models, indices):
    """Average the raw predicted IFC matrix across models, then run Phonopy once."""
    all_errors = []
    for i in indices:
        s = raw_samples[i]
        ifc_preds = []
        for model in models:
            model.eval()
            with torch.no_grad():
                bond_batch = Batch.from_data_list([bond_graphs[i]]).to(device)
                atom_batch = Batch.from_data_list([atom_graphs[i]]).to(device)
                ifc_pred = model(bond_batch, atom_batch)[0].cpu().numpy()
            ifc_preds.append(ifc_pred)
        avg_ifc = np.mean(ifc_preds, axis=0)
        try:
            s['all_q_points_subset'] = [[0, 0, 0]]
            pred_freqs = predict_frequencies_with_phonopy(avg_ifc, s)
            pred_peak = float(np.max(np.abs(pred_freqs)))
            true_peak = float(np.max(s['y_phonon']))
            all_errors.append(abs(true_peak - pred_peak))
        except Exception:
            continue
    return float(np.mean(all_errors)) if all_errors else float('nan')


ensemble_freq_mae = evaluate_ensemble_frequency_level(ensemble_models, test_idx)
ensemble_ifc_mae = evaluate_ensemble_ifc_level(ensemble_models, test_idx)

best_solo_mae = min(info['solo_test_freq_mae'] for info in ensemble_info)

print("=== Final comparison (test set) ===")
print(f"  v4 baseline (notebook 15, no tuning)         : 0.909 THz")
print(f"  Best single tuned model                      : {best_solo_mae:.4f} THz")
print(f"  Ensemble (frequency-level averaging)         : {ensemble_freq_mae:.4f} THz")
print(f"  Ensemble (IFC-level averaging)                : {ensemble_ifc_mae:.4f} THz")

/tmp/ipykernel_58/3431193008.py:10: DeprecationWarning: get_qpoints_dict() is deprecated. Use the qpoints property to access the Qpoints result object; its frequencies, eigenvectors, group_velocities, and dynamical_matrices attributes replace the dict keys.
  freqs = phonon.get_qpoints_dict()['frequencies']


=== Final comparison (test set) ===
  v4 baseline (notebook 15, no tuning)         : 0.909 THz
  Best single tuned model                      : 0.8903 THz
  Ensemble (frequency-level averaging)         : 0.8936 THz
  Ensemble (IFC-level averaging)                : 0.8941 THz


## جمع‌بندی و مراحل بعدی

این notebook دو مسیر بهبود فاز ۴ نقشه‌راه را روی معماری بدون‌تغییر v4 آزمایش کرد:

1. **Hyperparameter Tuning (Optuna)**: جست‌وجو روی نرخ یادگیری، وزن‌های Loss فیزیکی،
   اندازه‌ی hidden dim، تعداد لایه‌ها، dropout و batch size، با معیار مستقیم MAE فرکانس
   واقعی (نه IFC MSE) — هم برای انتخاب بهترین ترکیب و هم برای pruning trial های ضعیف.
2. **Ensemble**: آموزش کامل top-K ترکیب برتر با seed های متفاوت، و مقایسه‌ی دو استراتژی
   میانگین‌گیری (سطح فرکانس در برابر سطح IFC).

### ثبت در Obsidian (بعد از اجرا روی Kaggle)
- بهترین ترکیب هایپرپارامتر و MAE نهایی (`08 - نقشه راه مقاله.md`)
- جدول مقایسه‌ای v4 در برابر best-tuned در برابر ensemble (هر دو نوع averaging)
- در صورت بهبود معنادار: بررسی این‌که کدام هایپرپارامترها بیشترین تاثیر را داشتند
  (با `optuna.visualization.plot_param_importances`، قابل افزودن در نسخه‌ی بعدی اگر
  matplotlib/plotly در دسترس Kaggle باشد)

### محدودیت‌های شناخته‌شده (به ارث رسیده از v4)
- مدل هنوز فقط بلوک سلول مرجع IFC را پیش‌بینی می‌کند؛ عدد نهایی «حد بالا»ی ممکن است،
  نه عملکرد کامل روی کل ماتریس سوپرسل.
- ارزیابی فرکانس فقط در نقطه‌ی Γ انجام می‌شود (برای سرعت)؛ گسترش به چند نقطه‌ی q می‌تواند
  ارزیابی دقیق‌تری از منحنی dispersion کامل بدهد.